<a href="https://colab.research.google.com/github/bharshithacs24-coder/AI-LAB/blob/main/1WN24CS059_WEEK9_RESOLUTION_OF_FOL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from itertools import combinations
print("name: B Harshitha Pai   USN:1WN24CS059")
def is_var(x):
    return isinstance(x, str) and x.islower()

def unify(x, y, theta=None):
    if theta is None:
        theta = {}
    if x == y:
        return theta
    if is_var(x):
        return unify_var(x, y, theta)
    if is_var(y):
        return unify_var(y, x, theta)
    if isinstance(x, tuple) and isinstance(y, tuple) and len(x) == len(y):
        for a, b in zip(x, y):
            theta = unify(a, b, theta)
            if theta is None:
                return None
        return theta
    return None

def unify_var(var, x, theta):
    if var in theta:
        return unify(theta[var], x, theta)
    if is_var(x) and x in theta:
        return unify(var, theta[x], theta)
    theta[var] = x
    return theta

def substitute(clause, theta):
    new_clause = set()
    for neg, pred, args in clause:
        new_args = []
        for a in args:
            while is_var(a) and a in theta:
                a = theta[a]
            new_args.append(a)
        new_clause.add((neg, pred, tuple(new_args)))
    return frozenset(new_clause)

def standardize(clause, n):
    mapping = {}
    new_clause = set()
    for neg, pred, args in clause:
        new_args = []
        for a in args:
            if is_var(a):
                mapping.setdefault(a, f"{a}_{n}")
                new_args.append(mapping[a])
            else:
                new_args.append(a)
        new_clause.add((neg, pred, tuple(new_args)))
    return frozenset(new_clause)

def fmt(clause):
    if not clause:
        return "{}"
    return " v ".join(f"{'~' if neg else ''}{pred}({', '.join(args)})" for neg, pred, args in sorted(clause))

def to_cnf(formulas):
    cnf = []
    for f in formulas:
        if f[0] == "IMP":
            lhs, rhs = f[1], f[2]
            clause = [(not neg, p, a) for neg, p, a in lhs] + list(rhs)
            cnf.append(frozenset(clause))
        elif f[0] == "AND":
            for lit in f[1]:
                cnf.append(frozenset([lit]))
    return cnf

def resolve(ci, cj):
    resolvents = []
    for li in ci:
        for lj in cj:
            if li[1] == lj[1] and li[0] != lj[0]:
                theta = unify(li[2], lj[2])
                if theta is not None:
                    rest = (ci - {li}) | (cj - {lj})
                    resolvents.append(substitute(rest, theta))
    return resolvents

def print_proof(clause, parents):
    steps = []
    visited = set()
    def trace(c):
        if c in parents and c not in visited:
            visited.add(c)
            p1, p2 = parents[c]
            trace(p1)
            trace(p2)
            steps.append((p1, p2, c))
    trace(clause)
    print("\nResolution Steps:")
    for i, (p1, p2, res) in enumerate(steps, 1):
        print(f"  {i}. [{fmt(p1)}]  and  [{fmt(p2)}]  =>  {fmt(res)}")

def fol_resolution(premises, goal):
    clauses = to_cnf(premises)
    neg_goal = frozenset([(not goal[0], goal[1], goal[2])])
    clauses.append(neg_goal)

    print("CNF Clauses:")
    for i, c in enumerate(clauses, 1):
        print(f"  {i}. {fmt(c)}")

    seen = set(clauses)
    parents = {}
    counter = 1

    while True:
        new_clauses = []
        for ci, cj in combinations(clauses, 2):
            ci_s = standardize(ci, counter)
            cj_s = standardize(cj, counter + 1)
            counter += 2
            for res in resolve(ci_s, cj_s):
                if not res:
                    parents[res] = (ci, cj)
                    print_proof(res, parents)
                    return True
                if res not in seen:
                    seen.add(res)
                    parents[res] = (ci, cj)
                    new_clauses.append(res)
        if not new_clauses:
            return False
        clauses.extend(new_clauses)

premises = [
    ("IMP", [(False, "food", ("x",))], [(False, "likes", ("John", "x"))]),
    ("AND", [(False, "food", ("Apple",)), (False, "food", ("Vegetables",))]),
    ("IMP", [(False, "eats", ("x", "y")), (True, "killed", ("x",))], [(False, "food", ("y",))]),
    ("AND", [(False, "eats", ("Anil", "Peanuts")), (False, "alive", ("Anil",))]),
    ("IMP", [(False, "eats", ("Anil", "x"))], [(False, "eats", ("Harry", "x"))]),
    ("IMP", [(True, "killed", ("x",))], [(False, "alive", ("x",))]),
    ("IMP", [(False, "alive", ("x",))], [(True, "killed", ("x",))])
]

goal = (False, "likes", ("John", "Peanuts"))

print("\nProved:", fol_resolution(premises, goal))


name: B Harshitha Pai   USN:1WN24CS059
CNF Clauses:
  1. likes(John, x) v ~food(x)
  2. food(Apple)
  3. food(Vegetables)
  4. food(y) v killed(x) v ~eats(x, y)
  5. eats(Anil, Peanuts)
  6. alive(Anil)
  7. eats(Harry, x) v ~eats(Anil, x)
  8. alive(x) v killed(x)
  9. ~alive(x) v ~killed(x)
  10. ~likes(John, Peanuts)

Resolution Steps:
  1. [likes(John, x) v ~food(x)]  and  [~likes(John, Peanuts)]  =>  ~food(Peanuts)
  2. [food(y) v killed(x) v ~eats(x, y)]  and  [eats(Anil, Peanuts)]  =>  food(Peanuts) v killed(Anil)
  3. [alive(Anil)]  and  [~alive(x) v ~killed(x)]  =>  ~killed(Anil)
  4. [food(Peanuts) v killed(Anil)]  and  [~killed(Anil)]  =>  food(Peanuts)
  5. [~food(Peanuts)]  and  [food(Peanuts)]  =>  {}

Proved: True
